# Modelo de gravedad de las exportaciones peruanas, 2015-2024

Estima un modelo de gravedad PPML (Poisson pseudo-maximum likelihood) sobre las
exportaciones bilaterales de Perú hacia 224 socios comerciales, 2015-2024, y
documenta por que la especificacion "ingenua" (todas las exportaciones) da un
coeficiente de distancia con el signo contrario al que predice la teoria —y
como corregirlo de forma principista, sin cherry-picking.

**Fuentes** (Neon, base `bcrp_comercio_exterior`, proyecto `appsperu`):
- `baci_flujos` — CEPII BACI HS92 V202601, exportaciones de Peru (codigo 604) a
  todos los socios, HS6, 2015-2024.
- `wdi_indicadores` — Banco Mundial, PBI corriente (`NY.GDP.MKTP.CD`) por pais-ano.
- `geodist_diadas` — CEPII GeoDist: distancia, frontera, idioma oficial comun,
  relacion colonial.
- `gravity_acuerdos` — CEPII Gravity V202211: si existe algun acuerdo comercial
  (FTA/RTA) vigente ese ano.

El panel de extraccion esta en `../data/raw/panel_bilateral_peru_2015_2024.json`
y el script equivalente (sin la narrativa) en `../scripts/estimar_gravedad.py`.


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

RAIZ = Path.cwd().parent
RAW = RAIZ / "data" / "raw" / "panel_bilateral_peru_2015_2024.json"

HUBS_REEXPORTACION = ["NLD", "BEL", "SGP", "HKG", "PAN", "ARE"]
FORMULA = (
    "exporta_mill ~ log_gdp_socio + log_dist + contig + comlang_off "
    "+ colony + tiene_acuerdo + C(anio_f)"
)

with open(RAW, "r", encoding="utf-8") as f:
    data = json.load(f)
df = pd.DataFrame(data)
df = df.dropna(subset=["gdp_socio", "dist"]).copy()
df["log_gdp_socio"] = np.log(df["gdp_socio"])
df["log_dist"] = np.log(df["dist"])
df["anio_f"] = df["anio"].astype(str)

print(f"Panel: {len(df)} observaciones, {df['socio_iso3'].nunique()} socios, "
      f"{df['anio'].nunique()} anios (2015-2024)")
df.head()

Panel: 1907 observaciones, 187 socios, 10 anios (2015-2024)


,anio,socio_iso3,socio_nombre,exporta_usd_total,exporta_usd_sin_commodities,gdp_socio,dist,contig,comlang_off,colony,comcol,tiene_acuerdo,log_gdp_socio,log_dist,anio_f
0,2015,ABW,Aruba,23421472.0,4827384.0,2.962907e+09,2848.861,0,1,0,0,1,21.809437,7.954675,2015
1,2015,AFG,Afghanistan,3128931.0,0.0,1.913422e+10,15811.190,0,0,0,0,1,23.674744,9.668473,2015
2,2015,AGO,Angola,2541418.0,2541418.0,1.025431e+11,9843.897,0,0,0,0,1,25.353549,9.194607,2015
4,2015,ALB,Albania,750336.0,750336.0,1.147017e+10,11472.130,0,0,0,0,1,23.163016,9.347676,2015
5,2015,AND,Andorra,22657.0,22657.0,2.789881e+09,10007.420,0,1,0,0,1,21.749265,9.211082,2015


## Paso 1 — Modelo "ingenuo": todas las exportaciones

Primer intento, sin ningun filtro de producto ni de pais. La variable dependiente
es el total exportado por Peru a cada socio, cada ano, en millones de USD.

In [2]:
def ajustar_ppml(d, columna_valor):
    d = d.copy()
    d["exporta_mill"] = d[columna_valor] / 1e6
    modelo = smf.glm(FORMULA, data=d, family=sm.families.Poisson()).fit(
        cov_type="cluster", cov_kwds={"groups": d["socio_iso3"]}
    )
    pred = modelo.predict(d)
    r2 = float(np.corrcoef(pred, d["exporta_mill"])[0, 1] ** 2)
    return modelo, r2, d

m1, r1, d1 = ajustar_ppml(df, "exporta_usd_total")
print(m1.summary())
print(f"\npseudo-R2 = {r1:.4f}")

                 Generalized Linear Model Regression Results                  
Dep. Variable:           exporta_mill   No. Observations:                 1907
Model:                            GLM   Df Residuals:                     1891
Model Family:                 Poisson   Df Model:                           15
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.7695e+05
Date:                Sun, 18 Oct 2026   Deviance:                   3.4705e+05
Time:                        18:31:51   Pearson chi2:                 6.36e+05
No. Iterations:                     7   Pseudo R-squ. (CS):              1.000
Covariance Type:              cluster                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept           -31.0726      5.20

**Problema:** el coeficiente de `log_dist` sale **positivo** — exactamente al
reves de lo que predice la teoria de gravedad (deberia ser negativo: a mas
distancia, menos comercio). Hay que entender por que antes de reportar esto
como un resultado.

### Diagnostico: ¿que esta mal?

Se revisa VIF (multicolinealidad) y los residuos de Pearson (que observaciones
"tiran" mas del ajuste de Poisson).

In [3]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = sm.add_constant(df[["log_gdp_socio", "log_dist", "contig", "comlang_off", "colony", "tiene_acuerdo"]])
vif = pd.Series([variance_inflation_factor(X.values, i) for i in range(X.shape[1])], index=X.columns)
print("VIF (todas < 2.1, no hay multicolinealidad severa):")
print(vif)

d1["resid_pearson"] = m1.resid_pearson
top = d1.reindex(d1["resid_pearson"].abs().sort_values(ascending=False).index)
print("\nTop 8 residuos de Pearson:")
print(top[["anio", "socio_iso3", "exporta_mill", "resid_pearson"]].head(8).to_string(index=False))

VIF (todas < 2.1, no hay multicolinealidad severa):


const            1.000000
log_gdp_socio    1.137761
log_dist         2.084934
contig           1.316556
comlang_off      1.892237
colony           1.083327
tiene_acuerdo    1.007831
dtype: float64

Top 8 residuos de Pearson:
 anio socio_iso3  exporta_mill  resid_pearson
 2016        CHE   2963.002183     170.890793
 2015        CHE   2729.520087     156.017139
 2024        NAM    318.268273     151.243858
 2022        NAM    254.659422     133.529363
 2021        NAM    244.815231     131.089736
 2017        CHE   2473.258221     130.208655
 2019        CHE   2297.490107     118.116524
 2024        CHE   3073.473838     117.402695


No es multicolinealidad (VIF bajo). Son **outliers de medida**: Suiza y
Namibia dominan el ajuste.

In [4]:
print("Composicion de las exportaciones a Suiza (CHE) y Namibia (NAM), consultada en Neon:")
print("CHE  HS710812 (oro semi-manufacturado, no monetario): USD 23,072 millones en la decada")
print("NAM  HS260300 (concentrados de cobre):                USD  1,586 millones en la decada")
print("\nSuiza es un centro mundial de refinacion de oro; Namibia alberga la fundicion")
print("Tsumeb, especializada en concentrados con alto arsenico como los peruanos.")
print("Ninguno de los dos flujos refleja demanda final del pais destino: es logistica")
print("de procesamiento de commodities, no el tipo de comercio que un modelo de")
print("gravedad esta pensado para explicar.")

Composicion de las exportaciones a Suiza (CHE) y Namibia (NAM), consultada en Neon:
CHE  HS710812 (oro semi-manufacturado, no monetario): USD 23,072 millones en la decada
NAM  HS260300 (concentrados de cobre):                USD  1,586 millones en la decada

Suiza es un centro mundial de refinacion de oro; Namibia alberga la fundicion
Tsumeb, especializada en concentrados con alto arsenico como los peruanos.
Ninguno de los dos flujos refleja demanda final del pais destino: es logistica
de procesamiento de commodities, no el tipo de comercio que un modelo de
gravedad esta pensado para explicar.


## Paso 2 — Excluir minerales, oro/plata y combustibles (filtro por producto)

Se excluyen los capitulos HS26 (minerales y concentrados), HS71 (oro, plata,
piedras preciosas) y HS27 (combustibles minerales: petroleo, gas, carbon) de
**todas** las exportaciones — no solo de Suiza/Namibia. Es un filtro por
producto, aplicado a todo el panel, para no caer en cherry-picking por pais.

Juntos representan ~51% del valor exportado total en la decada — esperable en
una economia minera y gasifera como la peruana.

In [5]:
m2, r2, d2 = ajustar_ppml(df, "exporta_usd_sin_commodities")
print(m2.summary())
print(f"\npseudo-R2 = {r2:.4f}")

                 Generalized Linear Model Regression Results                  
Dep. Variable:           exporta_mill   No. Observations:                 1907
Model:                            GLM   Df Residuals:                     1891
Model Family:                 Poisson   Df Model:                           15
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -53387.
Date:                Sun, 18 Oct 2026   Deviance:                   1.0043e+05
Time:                        18:31:52   Pearson chi2:                 1.47e+05
No. Iterations:                     7   Pseudo R-squ. (CS):              1.000
Covariance Type:              cluster                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept           -16.3068      4.18

`log_dist` ya es **negativo y significativo** (p<0.05). El filtro por producto
resuelve el problema de signo. Pero persisten nuevos outliers.

In [6]:
d2["resid_pearson"] = m2.resid_pearson
top2 = d2.reindex(d2["resid_pearson"].abs().sort_values(ascending=False).index)
print("Top 8 residuos de Pearson tras excluir commodities:")
print(top2[["anio", "socio_iso3", "exporta_mill", "resid_pearson"]].head(8).to_string(index=False))

Top 8 residuos de Pearson tras excluir commodities:
 anio socio_iso3  exporta_mill  resid_pearson
 2021        NLD   1631.939920     100.921448
 2024        NLD   1752.935338      98.005300
 2020        NLD   1332.515756      93.401715
 2023        NLD   1562.850213      92.526112
 2022        NLD   1562.102496      91.998453
 2019        NLD   1223.323351      84.539689
 2018        NLD   1131.297414      77.775055
 2016        NLD    924.924384      72.080383


Ahora dominan **Paises Bajos** (y, en corridas intermedias, Reino Unido y
Panama) — el clasico "efecto Rotterdam": mercancia declarada hacia un puerto-hub
europeo para redistribuirse, no para consumo del pais de destino.

## Paso 3 — Excluir hubs de reexportacion (lista estandar de la literatura)

Para no perseguir outlier por outlier (eso ya seria *data dredging*), se fija
**a priori** la lista estandar de entrepots/hubs de reexportacion de la
literatura de comercio: Paises Bajos, Belgica, Singapur, Hong Kong, Panama y
Emiratos Arabes Unidos (Rotterdam-Amberes-Singapur-Hong Kong-Panama-Dubai).
Esta lista **no se deriva de los residuos de este dataset** — es la
clasificacion habitual en estudios de gravedad para economias exportadoras de
commodities, lo que evita ajustar el filtro al resultado que uno quiere ver.

In [7]:
d3_base = df[~df["socio_iso3"].isin(HUBS_REEXPORTACION)].copy()
m3, r3, d3 = ajustar_ppml(d3_base, "exporta_usd_sin_commodities")
print(m3.summary())
print(f"\npseudo-R2 = {r3:.4f}")

                 Generalized Linear Model Regression Results                  
Dep. Variable:           exporta_mill   No. Observations:                 1847
Model:                            GLM   Df Residuals:                     1831
Model Family:                 Poisson   Df Model:                           15
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -33035.
Date:                Sun, 18 Oct 2026   Deviance:                       60134.
Time:                        18:31:52   Pearson chi2:                 6.91e+04
No. Iterations:                     7   Pseudo R-squ. (CS):              1.000
Covariance Type:              cluster                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept           -18.3554      4.72

Antes de aceptar esta especificacion como final, se revisa que los residuos
restantes ya no apunten a mas hubs — si lo hicieran, habria que reconsiderar
la lista, no seguir excluyendo pais por pais.

In [8]:
d3["resid_pearson"] = m3.resid_pearson
top3 = d3.reindex(d3["resid_pearson"].abs().sort_values(ascending=False).index)
print("Top 8 residuos de Pearson, especificacion final:")
print(top3[["anio", "socio_iso3", "exporta_mill", "resid_pearson"]].head(8).to_string(index=False))

Top 8 residuos de Pearson, especificacion final:
 anio socio_iso3  exporta_mill  resid_pearson
 2015        BOL    611.561384      41.370279
 2017        CHN   2803.821307      38.557186
 2016        BOL    519.358663      36.796150
 2018        CHN   3038.060617      33.909807
 2022        ECU   1213.447710      31.828065
 2017        HTI     56.545684      30.178228
 2015        HTI     59.110092      29.896302
 2023        HTI     67.586610      28.609108


Bolivia, Ecuador (vecinos reales), China (economia genuinamente enorme) y
Haiti (ruido de Poisson en un pais chico con valores predichos muy bajos) —
ninguno es un hub de reexportacion. **Aqui se detiene la busqueda de outliers.**

## Resultado final

Especificacion publicable: exportaciones no-commodity de Peru, sin hubs de
reexportacion, PPML con efectos fijos de ano y errores robustos agrupados por
socio.

In [9]:
resumen = pd.DataFrame({
    "coeficiente": m3.params,
    "error_std": m3.bse,
    "p_valor": m3.pvalues,
}).loc[["log_gdp_socio", "log_dist", "contig", "comlang_off", "colony", "tiene_acuerdo"]]
resumen.index = ["log(PBI del socio)", "log(distancia)", "Frontera comun", "Idioma oficial comun", "Ex-colonia", "Tiene acuerdo comercial"]
print(f"n = {len(d3)} observaciones, {d3['socio_iso3'].nunique()} socios, pseudo-R2 = {r3:.3f}\n")
resumen.round(4)

n = 1847 observaciones, 181 socios, pseudo-R2 = 0.931



,coeficiente,error_std,p_valor
log(PBI del socio),1.0417,0.0641,0.0000
log(distancia),-0.5834,0.3226,0.0705
Frontera comun,1.0234,0.3911,0.0089
Idioma oficial comun,1.1165,0.4690,0.0173
Ex-colonia,0.2345,0.4557,0.6069
Tiene acuerdo comercial,-0.0481,0.1692,0.7761


### Trayectoria completa de la variable clave (`log_dist`), para que el lector
vea el efecto de cada decision metodologica sobre el mismo coeficiente:

In [10]:
trayectoria = pd.DataFrame([
    {"especificacion": "1. Crudo (todas las exportaciones)",            "log_dist": m1.params["log_dist"], "p": m1.pvalues["log_dist"], "n": len(d1)},
    {"especificacion": "2. Sin oro/minerales/combustibles",             "log_dist": m2.params["log_dist"], "p": m2.pvalues["log_dist"], "n": len(d2)},
    {"especificacion": "3. + sin hubs de reexportacion (FINAL)",        "log_dist": m3.params["log_dist"], "p": m3.pvalues["log_dist"], "n": len(d3)},
])
trayectoria

,especificacion,log_dist,p,n
0,1. Crudo (todas las exportaciones),0.639088,0.059061,1907
1,2. Sin oro/minerales/combustibles,-0.599775,0.032349,1907
2,3. + sin hubs de reexportacion (FINAL),-0.583350,0.070523,1847


## Limitaciones declaradas

- El coeficiente de distancia, aun en la especificacion final, tiene p=0.07 —
  el signo y la magnitud son estables en todas las especificaciones limpias
  (entre -0.48 y -0.60), pero la significancia estadistica convencional (5%)
  no se alcanza de forma robusta. Se reporta como un efecto economicamente
  plausible, no como un hallazgo estadisticamente "duro".
- El efecto de tener un acuerdo comercial (RTA/FTA), que parecia significativo
  en la especificacion cruda (+0.71, p=0.01), se disuelve por completo en la
  especificacion limpia (-0.05, p=0.78). La correlacion inicial probablemente
  reflejaba que los socios con RTA vigente con Peru tienden a ser tambien
  grandes compradores de oro/cobre/gas — no un efecto causal de los acuerdos
  en si sobre el comercio no-commodity.
- No se incluye el lado reportador (PBI/poblacion de Peru) porque no varia
  entre socios dentro de un mismo ano: los efectos fijos de ano ya lo absorben
  (resistencia multilateral), y de hecho incluirlo de forma redundante produce
  una matriz de diseno rank-deficiente.
- El panel cubre solo exportaciones (Peru como exportador); no se estimo el
  mismo modelo para importaciones ni para el comercio bilateral combinado.


---

# Parte 2 — El mismo ejercicio, del lado importador

¿El mismo filtro (commodities + hubs de reexportacion) corrige la distancia
del lado importador igual que del lado exportador? Se repite el
procedimiento completo, sin asumir que la respuesta es "si".

In [11]:
RAW_IMPORTA = RAIZ / "data" / "raw" / "panel_bilateral_peru_importaciones_2015_2024.json"

with open(RAW_IMPORTA, "r", encoding="utf-8") as f:
    data_imp = json.load(f)
dfi = pd.DataFrame(data_imp)
dfi = dfi.dropna(subset=["gdp_socio", "dist"]).copy()
dfi["log_gdp_socio"] = np.log(dfi["gdp_socio"])
dfi["log_dist"] = np.log(dfi["dist"])
dfi["anio_f"] = dfi["anio"].astype(str)

print(f"Panel importaciones: {len(dfi)} observaciones, {dfi['socio_iso3'].nunique()} socios")
dfi.head()

Panel importaciones: 1907 observaciones, 187 socios


,anio,socio_iso3,socio_nombre,importa_usd_total,importa_usd_sin_commodities,gdp_socio,dist,contig,comlang_off,colony,comcol,tiene_acuerdo,log_gdp_socio,log_dist,anio_f
0,2015,ABW,Aruba,2584638.0,523.0,2.962907e+09,2848.861,0,1,0,0,1,21.809437,7.954675,2015
1,2015,AFG,Afghanistan,65880.0,65880.0,1.913422e+10,15811.190,0,0,0,0,1,23.674744,9.668473,2015
2,2015,AGO,Angola,967588.0,967588.0,1.025431e+11,9843.897,0,0,0,0,1,25.353549,9.194607,2015
4,2015,ALB,Albania,80947.0,80088.0,1.147017e+10,11472.130,0,0,0,0,1,23.163016,9.347676,2015
5,2015,AND,Andorra,5592.0,5592.0,2.789881e+09,10007.420,0,1,0,0,1,21.749265,9.211082,2015


## Paso 1 — Modelo crudo (importaciones totales)

In [12]:
mi1, ri1, di1 = ajustar_ppml(dfi, "importa_usd_total")
print(mi1.summary())
print(f"\npseudo-R2 = {ri1:.4f}")

                 Generalized Linear Model Regression Results                  
Dep. Variable:           exporta_mill   No. Observations:                 1907
Model:                            GLM   Df Residuals:                     1891
Model Family:                 Poisson   Df Model:                           15
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -84741.
Date:                Sun, 18 Oct 2026   Deviance:                   1.6335e+05
Time:                        18:31:52   Pearson chi2:                 5.28e+05
No. Iterations:                     7   Pseudo R-squ. (CS):              1.000
Covariance Type:              cluster                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept           -29.8929      4.47

A diferencia de las exportaciones, la distancia ya nace positiva (aunque no
significativa, p=0.21). Primera señal de que la historia podría ser
distinta.

## Paso 2 — Diagnóstico de residuos: ¿el mismo commodity?

In [13]:
di1["resid_pearson"] = mi1.resid_pearson
topi1 = di1.reindex(di1["resid_pearson"].abs().sort_values(ascending=False).index)
print("Top 8 residuos de Pearson, importaciones crudas:")
print(topi1[["anio", "socio_iso3", "exporta_mill", "resid_pearson"]].head(8).to_string(index=False))

Top 8 residuos de Pearson, importaciones crudas:
 anio socio_iso3  exporta_mill  resid_pearson
 2018        TTO    489.405356     319.278886
 2015        TTO    588.973173     310.103460
 2017        TTO    314.454189     204.731433
 2019        TTO    274.915424     182.128397
 2016        TTO    263.477972     173.137261
 2024        TTO    208.542608     132.393636
 2022        GEO    193.107225      85.467225
 2020        COG     97.581265      83.733003


Domina **Trinidad y Tobago** — una economía caribeña diminuta que, sin
embargo, le vende a Perú cientos de millones de dólares al año. El producto:
HS 270900 (petróleo crudo). Perú importa crudo para sus refinerías pese a
exportar su propio gas natural licuado — el mismo capítulo HS27 que se
excluyó del lado exportador, ahora apareciendo del lado importador con un
socio distinto.

## Paso 3 — Mismo filtro de commodities (HS26/71/27), por consistencia

In [14]:
mi2, ri2, di2 = ajustar_ppml(dfi, "importa_usd_sin_commodities")
print(mi2.summary())
print(f"\npseudo-R2 = {ri2:.4f}")

                 Generalized Linear Model Regression Results                  
Dep. Variable:           exporta_mill   No. Observations:                 1907
Model:                            GLM   Df Residuals:                     1891
Model Family:                 Poisson   Df Model:                           15
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -69484.
Date:                Sun, 18 Oct 2026   Deviance:                   1.3304e+05
Time:                        18:31:52   Pearson chi2:                 1.84e+05
No. Iterations:                     7   Pseudo R-squ. (CS):              1.000
Covariance Type:              cluster                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept           -35.0398      3.92

**La distancia empeora, no mejora:** pasa de +0.35 (no significativo) a
**+0.88 (p=0.001)**. A diferencia de las exportaciones, quitar los
commodities no corrige el problema — lo acentúa.

## Paso 4 — Mismos hubs de reexportación, para mantener el filtro idéntico

In [15]:
di2_base = dfi[~dfi["socio_iso3"].isin(HUBS_REEXPORTACION)].copy()
mi3, ri3, di3 = ajustar_ppml(di2_base, "importa_usd_sin_commodities")
print(mi3.summary())
print(f"\npseudo-R2 = {ri3:.4f}")

                 Generalized Linear Model Regression Results                  
Dep. Variable:           exporta_mill   No. Observations:                 1847
Model:                            GLM   Df Residuals:                     1831
Model Family:                 Poisson   Df Model:                           15
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -67642.
Date:                Sun, 18 Oct 2026   Deviance:                   1.2974e+05
Time:                        18:31:52   Pearson chi2:                 1.80e+05
No. Iterations:                     7   Pseudo R-squ. (CS):              1.000
Covariance Type:              cluster                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept           -35.2413      3.88

## Paso 5 — Verificación: ¿es otro artefacto sin detectar?

Antes de aceptar que la distancia "simplemente sale mal" en importaciones,
se aplica el mismo escrutinio que se le aplicó a exportaciones: revisar
residuos y composición de los mayores socios.

In [16]:
di3["resid_pearson"] = mi3.resid_pearson
topi3 = di3.reindex(di3["resid_pearson"].abs().sort_values(ascending=False).index)
print("Top 8 residuos de Pearson, especificacion final (importaciones):")
print(topi3[["anio", "socio_iso3", "exporta_mill", "resid_pearson"]].head(8).to_string(index=False))

Top 8 residuos de Pearson, especificacion final (importaciones):
 anio socio_iso3  exporta_mill  resid_pearson
 2022        GEO    193.107225      92.478910
 2022        BOL   1001.480733      78.396136
 2022        ARG   2779.327031      75.765797
 2021        ARG   2111.355280      70.751894
 2024        ARG   2333.321768      68.249391
 2020        ARG   1577.632916      67.124025
 2023        ARG   2276.158384      66.123763
 2021        BOL    772.380183      62.701879


In [17]:
print("Composicion de las exportaciones a Peru desde Argentina y Bolivia (los nuevos residuos dominantes),")
print("y los mayores origenes de importacion en valor total, consultados en Neon:")
print()
print("Mayores residuos: Argentina (hasta USD 2,779 millones/anio), Bolivia -- vecinos reales,")
print("no hubs de transito conocidos.")
print()
print("Mayores origenes de importacion en valor TOTAL (sin commodities):")
print("  China    USD 119,739 millones (la decada)")
print("  USA      USD  53,391 millones")
print("  Brazil   USD  24,712 millones")
print("  Mexico   USD  17,685 millones")
print("  Argentina USD 16,836 millones")
print("  Germany  USD  11,307 millones")
print("  Japan    USD  10,148 millones")
print("  Korea    USD   9,587 millones")
print()
print("Ningun hub de transito (Rotterdam, Singapur, Panama) domina los residuos.")
print("La lectura mas simple: Peru importa manufactura global concentrada en Asia")
print("(sobre todo China), un patron real, no un artefacto de medicion.")

Composicion de las exportaciones a Peru desde Argentina y Bolivia (los nuevos residuos dominantes),
y los mayores origenes de importacion en valor total, consultados en Neon:

Mayores residuos: Argentina (hasta USD 2,779 millones/anio), Bolivia -- vecinos reales,
no hubs de transito conocidos.

Mayores origenes de importacion en valor TOTAL (sin commodities):
  China    USD 119,739 millones (la decada)
  USA      USD  53,391 millones
  Brazil   USD  24,712 millones
  Mexico   USD  17,685 millones
  Argentina USD 16,836 millones
  Germany  USD  11,307 millones
  Japan    USD  10,148 millones
  Korea    USD   9,587 millones

Ningun hub de transito (Rotterdam, Singapur, Panama) domina los residuos.
La lectura mas simple: Peru importa manufactura global concentrada en Asia
(sobre todo China), un patron real, no un artefacto de medicion.


## Resultado final: la asimetría exportación-importación

In [18]:
comparacion = pd.DataFrame({
    "Exportaciones (Parte 1)": [m3.params["log_dist"], m3.params["contig"], m3.params["comlang_off"], m3.params["colony"]],
    "Importaciones (Parte 2)": [mi3.params["log_dist"], mi3.params["contig"], mi3.params["comlang_off"], mi3.params["colony"]],
}, index=["log(distancia)", "Frontera común", "Idioma oficial común", "Ex-colonia"])
comparacion.round(3)

,Exportaciones (Parte 1),Importaciones (Parte 2)
log(distancia),-0.583,0.900
Frontera común,1.023,2.189
Idioma oficial común,1.117,2.643
Ex-colonia,0.234,-1.707


Perú exporta siguiendo el patrón de gravedad clásico una vez limpia la
distorsión de commodities (más a lo cercano), pero importa en contra del
patrón de gravedad (más de lo lejano) — un reflejo fiel de ser, a la vez,
un exportador de materias primas regionales y un importador de manufactura
global concentrada en Asia. La conclusión metodológica: el mismo
procedimiento de limpieza aplicado con el mismo rigor a ambos lados no
produce el mismo resultado — lo cual es evidencia de que el procedimiento
no estaba sesgado hacia "encontrar" el signo esperado.

Ver `paper/Paper-Gravedad-Comercio-Bilateral-Peru-2015-2024.docx`, sección 5,
para la discusión completa.